# SCENIC Regulon Exploration — JunB patient vs Healthy Controls

Explores pySCENIC output from the JunB scRNA-seq pipeline:
- **29 regulons** recovered (6000 HVGs + 2 cisTarget databases)
- 163,807 PBMCs — 7 HD + 1 PID7 (JunB patient)
- AP1/JunB-related regulons: **ATF3(+)**, **FOSL2(+)**, **MAFB(+)**, **NFE2(+)**
- JUNB GRN adjacencies (3,249 edges) available even though cisTarget filtered it out

In [ ]:
import pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=100, facecolor='white', frameon=False)

SCENIC_DIR = '../results_HVG6000/scenic'
# condition labels in adata.obs
PALETTE = {'HD': '#4878CF', 'PID7': '#D65F5F'}

## 1. Load data

In [ ]:
adata = sc.read_h5ad(f'{SCENIC_DIR}/adata_scenic.h5ad')

with open(f'{SCENIC_DIR}/regulons.pkl', 'rb') as fh:
    regulons = pickle.load(fh)

auc_mtx = pd.read_csv(f'{SCENIC_DIR}/auc_matrix.csv', index_col=0)
rss     = pd.read_csv(f'{SCENIC_DIR}/rss_scores.csv',  index_col=0).fillna(0)
adj     = pd.read_csv(f'{SCENIC_DIR}/adj.tsv', sep='\t')

regulon_names = list(adata.uns['scenic_regulon_names'])

# Align AUC matrix to adata cell order, then attach to obs for scanpy plotting
auc_aligned = auc_mtx.reindex(adata.obs_names)
for reg in regulon_names:
    adata.obs[reg] = auc_aligned[reg].values

# AP1 / JunB family regulons present in this run
AP1_RELATED = [r for r in regulon_names
               if any(k in r for k in ['ATF','JUN','FOS','BATF','MAF','NFE2'])]

print(f'Cells: {adata.n_obs:,}  |  Genes: {adata.n_vars}  |  Regulons: {len(regulons)}')
print(f'Conditions: {adata.obs["condition"].value_counts().to_dict()}')
print(f'\nAP1/JunB-related regulons: {AP1_RELATED}')
print(f'\nAll regulons:')
for r in sorted(regulons, key=lambda x: x.name):
    flag = ' <--' if r.name in AP1_RELATED else ''
    print(f'  {r.name:18s}  {len(r.genes):3d} targets{flag}')

## 2. AUC score distribution per regulon

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
auc_mtx[regulon_names].plot.box(ax=ax, rot=45)
ax.set_ylabel('AUC score')
ax.set_title('Regulon AUC score distributions')
# Highlight AP1-related
for tick in ax.get_xticklabels():
    if tick.get_text() in AP1_RELATED:
        tick.set_color('#D65F5F')
        tick.set_fontweight('bold')
plt.tight_layout()
plt.show()

## 3. UMAP — cell type, condition, and regulon activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sc.pl.umap(adata, color='cell_type', legend_loc='on data', legend_fontsize=6,
           title='Cell type', ax=axes[0], show=False, frameon=False)
sc.pl.umap(adata, color='condition', palette=PALETTE, title='Condition',
           ax=axes[1], show=False, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# All regulons on UMAP — AP1-related ones first
ordered = AP1_RELATED + [r for r in regulon_names if r not in AP1_RELATED]
ncols = 5
nrows = int(np.ceil(len(ordered) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3))
axes = axes.flatten()

for i, reg in enumerate(ordered):
    title = reg + (' ★' if reg in AP1_RELATED else '')
    sc.pl.umap(adata, color=reg, color_map='YlOrRd', title=title,
               ax=axes[i], show=False, frameon=False, colorbar_loc='right', size=0.5)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Regulon AUC activity (★ = AP1/JunB family)', y=1.01, fontsize=11)
plt.tight_layout()
plt.show()

## 4. Regulon Specificity Score (RSS) heatmap

In [ ]:
# Transpose: rows=regulons, cols=cell types
rss_T = rss.T
rss_sorted = rss_T.loc[rss_T.max(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(max(10, len(rss_sorted.columns) * 0.6),
                                max(7, len(rss_sorted) * 0.35)))
sns.heatmap(rss_sorted, cmap='viridis', linewidths=0.2, linecolor='grey',
            ax=ax, cbar_kws={'label': 'RSS'})
ax.set_title('Regulon Specificity Score per cell type (sorted by max RSS)')
ax.set_xlabel('Cell type')
ax.set_ylabel('Regulon')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
# Highlight AP1-related regulons
for tick in ax.get_yticklabels():
    if tick.get_text() in AP1_RELATED:
        tick.set_color('#D65F5F')
        tick.set_fontweight('bold')
plt.tight_layout()
plt.show()

print('\nTop cell type per regulon:')
print(rss.idxmax(axis=0).to_string())

## 5. HD vs PID7 — regulon activity comparison

In [ ]:
hd_mask  = adata.obs['condition'] == 'HD'
pid_mask = adata.obs['condition'] == 'PID7'

summary = pd.DataFrame({
    'HD_mean':   auc_mtx.reindex(adata.obs_names[hd_mask]).mean(),
    'PID7_mean': auc_mtx.reindex(adata.obs_names[pid_mask]).mean(),
})
summary['log2FC'] = np.log2(
    (summary['PID7_mean'] + 1e-6) / (summary['HD_mean'] + 1e-6)
)
summary = summary.sort_values('log2FC', ascending=False)

# Highlight AP1-related
print('Regulon AUC: PID7 vs HD (sorted by log2FC)')
print(summary.round(5).to_string())

fig, ax = plt.subplots(figsize=(6, max(4, len(summary) * 0.35)))
colors = ['#D65F5F' if v > 0 else '#4878CF' for v in summary['log2FC']]
bars = ax.barh(summary.index, summary['log2FC'], color=colors)
# Bold AP1 tick labels
for tick in ax.get_yticklabels():
    if tick.get_text() in AP1_RELATED:
        tick.set_fontweight('bold')
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('log2 FC (PID7 / HD)')
ax.set_title('Global regulon activity: PID7 vs HD')
plt.tight_layout()
plt.show()

In [ ]:
# Violin plots — all regulons, split by condition
ncols = 5
nrows = int(np.ceil(len(regulon_names) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 3))
axes = axes.flatten()

for i, reg in enumerate(ordered):   # AP1-related first
    plot_df = adata.obs[['condition', reg]].copy()
    sns.violinplot(data=plot_df, x='condition', y=reg,
                   palette=PALETTE, inner='quartile', cut=0,
                   ax=axes[i], linewidth=0.8, order=['HD', 'PID7'])
    title = reg + (' ★' if reg in AP1_RELATED else '')
    axes[i].set_title(title, fontsize=8,
                      color='#D65F5F' if reg in AP1_RELATED else 'black')
    axes[i].set_xlabel('')
    axes[i].set_ylabel('AUC', fontsize=7)
    axes[i].tick_params(axis='x', labelsize=7)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Regulon AUC: HD vs PID7  (★ = AP1/JunB family)', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Myeloid-only comparison
mono_types = [ct for ct in adata.obs['cell_type'].unique()
              if any(k in ct.lower() for k in ['monocyte','macrophage','dc','dendritic'])]
print('Myeloid cell types:', mono_types)

adata_mono = adata[adata.obs['cell_type'].isin(mono_types)]
fig, axes = plt.subplots(1, len(AP1_RELATED), figsize=(len(AP1_RELATED) * 3, 4))
if len(AP1_RELATED) == 1:
    axes = [axes]

for ax, reg in zip(axes, AP1_RELATED):
    plot_df = adata_mono.obs[['condition', reg]].copy()
    sns.violinplot(data=plot_df, x='condition', y=reg,
                   palette=PALETTE, inner='quartile', cut=0,
                   ax=ax, linewidth=0.8, order=['HD', 'PID7'])
    ax.set_title(reg, fontsize=9, color='#D65F5F')
    ax.set_xlabel('')

plt.suptitle('Myeloid cells — AP1/JunB regulon AUC: HD vs PID7', y=1.02)
plt.tight_layout()
plt.show()

## 6. AP1/JunB-related regulons deep-dive

In [ ]:
for reg_name in AP1_RELATED:
    r_obj = next(r for r in regulons if r.name == reg_name)
    targets = sorted(r_obj.genes)
    print(f'\n{reg_name}  ({len(targets)} targets)')
    print('  ' + ', '.join(targets))

In [ ]:
# UMAP for AP1-related regulons side by side
fig, axes = plt.subplots(1, len(AP1_RELATED), figsize=(len(AP1_RELATED) * 4, 4))
if len(AP1_RELATED) == 1:
    axes = [axes]

for ax, reg in zip(axes, AP1_RELATED):
    sc.pl.umap(adata, color=reg, color_map='RdYlBu_r', ax=ax,
               title=reg, show=False, frameon=False, colorbar_loc='right', size=0.5)
plt.suptitle('AP1/JunB-related regulon activity', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Dot plot of AP1-related regulon target genes across top cell types
for reg_name in AP1_RELATED:
    r_obj  = next(r for r in regulons if r.name == reg_name)
    genes  = [g for g in sorted(r_obj.genes) if g in adata.var_names][:20]
    top_cts = rss[reg_name].nlargest(8).index.tolist()
    adata_sub = adata[adata.obs['cell_type'].isin(top_cts)]
    if genes:
        sc.pl.dotplot(adata_sub, var_names=genes, groupby='cell_type',
                      standard_scale='var',
                      title=f'{reg_name} target genes — top 8 cell types by RSS',
                      figsize=(max(10, len(genes) * 0.7), 4))

## 7. JUNB expression and GRN adjacencies

In [ ]:
# JUNB expression on UMAP
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sc.pl.umap(adata, color='JUNB', color_map='RdYlBu_r', title='JUNB expression',
           ax=axes[0], show=False, frameon=False, colorbar_loc='right', size=0.5)
sc.pl.umap(adata, color='JUN', color_map='RdYlBu_r', title='JUN expression',
           ax=axes[1], show=False, frameon=False, colorbar_loc='right', size=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# JUNB violin: all cells and myeloid cells
# sc.pl.violin leaks title= into matplotlib in scanpy 1.11 — use seaborn directly
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

for ax, (adata_use, title) in zip(axes, [
    (adata,      'JUNB — all cells'),
    (adata_mono, 'JUNB — myeloid cells'),
]):
    junb_idx = list(adata_use.var_names).index('JUNB')
    X = adata_use.X
    expr = X[:, junb_idx].toarray().ravel() if hasattr(X, 'toarray') else np.asarray(X[:, junb_idx]).ravel()
    plot_df = pd.DataFrame({'condition': adata_use.obs['condition'].values, 'JUNB': expr})
    sns.violinplot(data=plot_df, x='condition', y='JUNB',
                   palette=PALETTE, inner='quartile', cut=0,
                   ax=ax, linewidth=0.8, order=['HD', 'PID7'])
    ax.set_title(title)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# JUNB GRN adjacencies
junb_adj = adj[adj['TF'] == 'JUNB'].sort_values('importance', ascending=False)
print(f'JUNB GRN edges: {len(junb_adj)}')
print(junb_adj.head(20).to_string(index=False))

In [ ]:
# Dot plot of top JUNB GRN targets
top_junb_targets = junb_adj.head(30)['target'].tolist()
top_junb_in_adata = [g for g in top_junb_targets if g in adata.var_names]
print(f'Top 30 JUNB targets in adata: {len(top_junb_in_adata)}')

show_genes = ['JUNB'] + top_junb_in_adata[:20]
sc.pl.dotplot(adata, var_names=show_genes, groupby='cell_type',
              standard_scale='var',
              title='Top JUNB GRN targets across cell types',
              figsize=(14, 6))

In [ ]:
# Overlap: JUNB GRN targets vs AP1 regulon target genes
junb_target_set = set(junb_adj['target'])
print('JUNB GRN target overlap with AP1/JunB regulons:')
for reg_name in AP1_RELATED:
    r_obj   = next(r for r in regulons if r.name == reg_name)
    reg_genes = set(r_obj.genes)
    overlap = junb_target_set & reg_genes
    pct = 100 * len(overlap) / len(reg_genes)
    print(f'  {reg_name}: {len(overlap)}/{len(reg_genes)} targets ({pct:.0f}%)')
    if overlap:
        print('    ', ', '.join(sorted(overlap)))

## 8. Mean AUC per cell type — clustered heatmap

In [ ]:
mean_auc = (
    adata.obs[['cell_type'] + regulon_names]
    .groupby('cell_type')[regulon_names]
    .mean()
)
mean_auc_z = mean_auc.apply(lambda x: (x - x.mean()) / (x.std() + 1e-8))

g = sns.clustermap(
    mean_auc_z,
    cmap='RdBu_r', center=0,
    figsize=(max(10, len(regulon_names) * 0.6), max(8, len(mean_auc) * 0.4)),
    linewidths=0.2,
    cbar_pos=(1.01, 0.3, 0.03, 0.4),
    dendrogram_ratio=0.1,
    xticklabels=True,
    yticklabels=True,
)
g.ax_heatmap.set_xlabel('')
g.ax_heatmap.set_ylabel('')
plt.setp(g.ax_heatmap.get_xticklabels(), rotation=45, ha='right', fontsize=8)
plt.setp(g.ax_heatmap.get_yticklabels(), fontsize=8)

# Bold AP1-related x-tick labels
for tick in g.ax_heatmap.get_xticklabels():
    if tick.get_text() in AP1_RELATED:
        tick.set_color('#D65F5F')
        tick.set_fontweight('bold')

g.fig.suptitle('Mean AUC per cell type (z-scored, red = AP1/JunB)', y=1.01)
plt.show()

## 9. PID7 vs HD AUC difference per cell type — AP1/JunB regulons

In [ ]:
obs = adata.obs[['cell_type', 'condition'] + AP1_RELATED].copy()

fig, axes = plt.subplots(1, len(AP1_RELATED),
                         figsize=(len(AP1_RELATED) * 5, max(5, len(obs['cell_type'].unique()) * 0.3)))
if len(AP1_RELATED) == 1:
    axes = [axes]

for ax, reg in zip(axes, AP1_RELATED):
    ct_cond = (
        obs.groupby(['cell_type', 'condition'])[reg]
        .mean()
        .unstack('condition')
        .assign(diff=lambda d: d.get('PID7', 0) - d.get('HD', 0))
        .sort_values('diff', ascending=False)
    )
    colors = ['#D65F5F' if v > 0 else '#4878CF' for v in ct_cond['diff']]
    ax.barh(ct_cond.index, ct_cond['diff'], color=colors)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('Mean AUC (PID7 − HD)')
    ax.set_title(reg, color='#D65F5F')
    ax.tick_params(axis='y', labelsize=7)

plt.suptitle('AP1/JunB regulon: PID7 vs HD per cell type', y=1.01)
plt.tight_layout()
plt.show()